# 🌾 Analyse des Productions Agricoles Mondiales
**Source :** FAO (Organisation des Nations Unies pour l'alimentation et l'agriculture)  
**Période :** 1961 – 2007  
**Auteur :** Ton prénom  
**Date :** 2024

---

## Objectifs de cette analyse

1. Explorer la structure du dataset FAO
2. Identifier les cultures les plus produites dans le monde
3. Analyser l'évolution de la production de blé (1961–2007)
4. Comparer la production de maïs entre les grandes régions du monde
5. Tirer des conclusions sur les tendances agricoles mondiales

---
## 1. Importation des bibliothèques

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Paramètres graphiques globaux
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['grid.linestyle'] = '--'

print('Bibliothèques chargées ✓')

---
## 2. Chargement et exploration du dataset

In [ ]:
# Chargement du fichier
df = pd.read_csv('JEU_DE_DONNEE.csv')

print(f'Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Colonnes   : {df.columns.tolist()}')

In [ ]:
# Aperçu des premières lignes
df.head(10)

In [ ]:
# Types de données et valeurs manquantes
print('--- Types de colonnes ---')
print(df.dtypes)
print()
print('--- Valeurs manquantes ---')
print(df.isnull().sum())

In [ ]:
# Résumé des valeurs uniques
print(f"Pays / zones    : {df['country_or_area'].nunique()}")
print(f"Cultures        : {df['category'].nunique()}")
print(f"Types de mesure : {df['element'].unique()}")
print(f"Années          : {int(df['year'].min())} → {int(df['year'].max())}")

> **Observation :** Le dataset contient plus de 2 millions de lignes couvrant 259 pays, 172 cultures et 47 ans de données. On note des valeurs manquantes dans la colonne `value` et `value_footnotes`.

---
## 3. Top 10 des cultures les plus produites

In [ ]:
# Filtrer : production mondiale, quantité uniquement
df_prod = df[
    (df['country_or_area'] == 'World +') &
    (df['element'] == 'Production Quantity')
].dropna(subset=['value']).copy()

# Calculer le top 10
top10 = (
    df_prod.groupby('category')['value']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top10.columns = ['Culture', 'Production totale (tonnes)']
top10.index = top10.index + 1
top10.index.name = 'Rang'
top10

In [ ]:
# Graphique : top 10 en barres horizontales
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2563EB' if i == 0 else '#93C5FD' for i in range(10)]
bars = ax.barh(
    top10['Culture'][::-1],
    top10['Production totale (tonnes)'][::-1] / 1e9,
    color=colors[::-1]
)

ax.set_title('Top 10 des cultures les plus produites (1961–2007)', fontsize=15, fontweight='bold')
ax.set_xlabel('Production totale (milliards de tonnes)')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f} Gt'))

plt.tight_layout()
plt.savefig('top10_cultures.png', dpi=150, bbox_inches='tight')
plt.show()

> **Observation :** Les céréales dominent la production mondiale. Le blé (wheat) se classe en 6ème position avec plus de 21 milliards de tonnes cumulées sur la période.

---
## 4. Évolution de la production mondiale de blé (1961–2007)

In [ ]:
# Filtrer blé mondial
df_ble = df[
    (df['category'] == 'wheat') &
    (df['country_or_area'] == 'World +') &
    (df['element'] == 'Production Quantity')
].dropna(subset=['value']).sort_values('year').copy()

df_ble['valeur_millions'] = df_ble['value'] / 1_000_000

print(f"{len(df_ble)} années de données (blé mondial)")
df_ble[['year', 'valeur_millions', 'unit']].head(5)

In [ ]:
# Graphique évolution blé
annees     = df_ble['year']
production = df_ble['valeur_millions']
moyenne    = production.mean()
idx_max    = production.idxmax()
idx_min    = production.idxmin()

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(annees, production, color='#2563EB', linewidth=2.5, marker='o', markersize=4)
ax.fill_between(annees, production, alpha=0.1, color='#2563EB')
ax.axhline(y=moyenne, color='orange', linestyle='--', linewidth=1.5, label=f'Moyenne : {moyenne:.0f} Mt')

ax.annotate(
    f"Max : {production[idx_max]:.0f} Mt\n({int(annees[idx_max])})",
    xy=(annees[idx_max], production[idx_max]),
    xytext=(annees[idx_max] - 9, production[idx_max] - 35),
    arrowprops=dict(arrowstyle='->', color='green'), color='green', fontsize=10
)
ax.annotate(
    f"Min : {production[idx_min]:.0f} Mt\n({int(annees[idx_min])})",
    xy=(annees[idx_min], production[idx_min]),
    xytext=(annees[idx_min] + 3, production[idx_min] + 40),
    arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10
)

ax.set_title('Production mondiale de blé (1961–2007)', fontsize=15, fontweight='bold')
ax.set_xlabel('Année')
ax.set_ylabel('Production (millions de tonnes)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f} Mt'))
ax.legend(fontsize=11)
ax.set_xlim(1961, 2007)

plt.tight_layout()
plt.savefig('ble_mondial.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistiques clés
debut     = production.iloc[0]
fin       = production.iloc[-1]
croissance = ((fin - debut) / debut) * 100

print(f"Production en 1961  : {debut:.0f} Mt")
print(f"Production en 2007  : {fin:.0f} Mt")
print(f"Maximum             : {production.max():.0f} Mt ({int(annees[idx_max])})")
print(f"Moyenne             : {moyenne:.0f} Mt")
print(f"Croissance totale   : +{croissance:.0f}% en 46 ans")

> **Observation :** La production mondiale de blé a progressé de **+172%** entre 1961 et 2007, passant de 222 Mt à 606 Mt. On observe une forte accélération dans les années 1960–1990 (Révolution verte), puis une stabilisation après 1990.

---
## 5. Comparaison de la production de maïs par région (1961–2007)

In [ ]:
# Régions à comparer
regions = ['Northern America +', 'Asia +', 'Europe +', 'South America +', 'Africa +']

df_maize = df[
    (df['category'] == 'maize') &
    (df['element'] == 'Production Quantity') &
    (df['country_or_area'].isin(regions))
].dropna(subset=['value']).copy()

df_maize['valeur_millions'] = df_maize['value'] / 1_000_000

# Tableau pivot
pivot = df_maize.pivot_table(
    index='year',
    columns='country_or_area',
    values='valeur_millions',
    aggfunc='sum'
)
pivot.columns = [c.replace(' +', '') for c in pivot.columns]
pivot.head()

In [ ]:
# Graphique courbes multi-régions
couleurs = {
    'Northern America': '#2563EB',
    'Asia'            : '#DC2626',
    'Europe'          : '#16A34A',
    'South America'   : '#D97706',
    'Africa'          : '#7C3AED'
}

fig, ax = plt.subplots(figsize=(13, 6))

for region in pivot.columns:
    ax.plot(
        pivot.index, pivot[region],
        label=region, linewidth=2.5,
        marker='o', markersize=3.5,
        color=couleurs.get(region, 'gray')
    )

ax.set_title('Production de maïs par région (1961–2007)', fontsize=15, fontweight='bold')
ax.set_xlabel('Année')
ax.set_ylabel('Production (millions de tonnes)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f} Mt'))
ax.set_xlim(1961, 2007)
ax.legend(fontsize=11, loc='upper left')

plt.tight_layout()
plt.savefig('maize_regions.png', dpi=150, bbox_inches='tight')
plt.show()

> **Observation :** L'Amérique du Nord domine largement la production mondiale de maïs (États-Unis = 1er producteur mondial). L'Asie connaît une forte croissance à partir des années 1980. L'Afrique et l'Amérique du Sud progressent nettement après 1990.

---
## 6. Conclusions

### Ce que l'analyse nous apprend :

| Constat | Détail |
|---|---|
| Les céréales dominent | Blé, maïs, riz = 3 des 4 premières cultures mondiales |
| Croissance spectaculaire du blé | +172% entre 1961 et 2007 (Révolution verte) |
| Domination nord-américaine du maïs | Les USA représentent la majorité de la production mondiale |
| Montée en puissance de l'Asie | Forte croissance agricole à partir des années 1980 |
| Stabilisation après 1990 | La croissance ralentit dans les pays développés |

### Prochaines étapes possibles :
- Analyse par pays (France, Chine, USA...)
- Corrélation entre surface cultivée et rendement
- **ACP** : regrouper les pays selon leur profil agricole → voir section 7

---
## 7. ACP — Analyse en Composantes Principales

**Objectif :** Regrouper les 219 pays selon leur profil agricole.

**Principe :** On a 10 cultures (= 10 dimensions). L'ACP les compresse en 2 dimensions pour pouvoir tracer un graphique. Les pays proches = profils similaires.

| Étape | Action |
|---|---|
| 1 | Construire un tableau `pays × cultures` |
| 2 | Standardiser (même échelle pour tous les pays) |
| 3 | Appliquer PCA (10 colonnes → 2) |
| 4 | Visualiser en 2D |

In [ ]:
# Installer si nécessaire : !pip install scikit-learn
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print('Bibliothèques ACP chargées ✓')

In [ ]:
# --- Étape 1 : Tableau pays × cultures ---

df_pays = df[
    (df['element'] == 'Production Quantity') &
    (~df['country_or_area'].str.contains(r'\+', na=False))
].dropna(subset=['value'])

cultures = [
    'cereals_total', 'roots_and_tubers_total', 'vegetables_melons_total',
    'fruit_excl_melons_total', 'oilcrops_primary', 'pulses_total',
    'wheat', 'maize', 'rice_paddy', 'sugar_cane'
]

df_agg = (
    df_pays[df_pays['category'].isin(cultures)]
    .groupby(['country_or_area', 'category'])['value']
    .sum()
    .reset_index()
)

# Une ligne = un pays, une colonne = une culture
tableau = df_agg.pivot_table(
    index='country_or_area',
    columns='category',
    values='value',
    aggfunc='sum'
).fillna(0)

print(f'Tableau : {tableau.shape[0]} pays × {tableau.shape[1]} cultures')
tableau.head()

In [ ]:
# --- Étape 2 : Standardisation ---
# Obligatoire : ramène toutes les valeurs à la même échelle
# (moyenne=0, écart-type=1) pour que la Chine n'écrase pas les petits pays

scaler = StandardScaler()
tableau_scaled = scaler.fit_transform(tableau)

print(f'Moyenne : {tableau_scaled.mean():.4f} (≈ 0)')
print(f'Écart-type : {tableau_scaled.std():.4f} (≈ 1)')

In [ ]:
# --- Étape 3 : ACP ---
# n_components=2 : compresse 10 cultures en 2 axes

pca = PCA(n_components=2)
coordonnees = pca.fit_transform(tableau_scaled)

var_pc1 = pca.explained_variance_ratio_[0] * 100
var_pc2 = pca.explained_variance_ratio_[1] * 100

print(f'PC1 : {var_pc1:.1f}% de variance expliquée')
print(f'PC2 : {var_pc2:.1f}% de variance expliquée')
print(f'Total conservé : {var_pc1 + var_pc2:.1f}% de l\'information originale')

In [ ]:
# --- Étape 4 : Graphique ACP ---

df_pca = pd.DataFrame({
    'pays': tableau.index,
    'PC1' : coordonnees[:, 0],
    'PC2' : coordonnees[:, 1]
})

grands = ['China', 'United States of America', 'India', 'Brazil',
          'France', 'Russian Federation', 'Australia', 'Argentina',
          'Germany', 'Canada', 'Pakistan', 'Indonesia']
df_pca['est_grand'] = df_pca['pays'].isin(grands)

fig, ax = plt.subplots(figsize=(14, 9))

petits = df_pca[~df_pca['est_grand']]
ax.scatter(petits['PC1'], petits['PC2'],
           color='lightsteelblue', alpha=0.5, s=30, label='Autres pays')

gds = df_pca[df_pca['est_grand']]
ax.scatter(gds['PC1'], gds['PC2'],
           color='#1E3A8A', s=100, zorder=5, label='Grands producteurs')

for _, row in gds.iterrows():
    nom = (row['pays']
           .replace('United States of America', 'USA')
           .replace('Russian Federation', 'Russie'))
    ax.annotate(nom, xy=(row['PC1'], row['PC2']),
                xytext=(row['PC1'] + 0.15, row['PC2'] + 0.15),
                fontsize=9, color='#1E3A8A', fontweight='bold')

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('ACP — Profils agricoles des pays du monde (1961–2007)',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel(f'PC1 ({var_pc1:.1f}% de variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({var_pc2:.1f}% de variance)', fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('acp_pays.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Bonus : Cercle des corrélations ---
# Montre quelles cultures influencent chaque axe

composantes = pd.DataFrame(
    pca.components_.T,
    index=cultures,
    columns=['PC1', 'PC2']
)

fig2, ax2 = plt.subplots(figsize=(9, 9))

for culture, row in composantes.iterrows():
    ax2.annotate('', xy=(row['PC1'], row['PC2']), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='#2563EB', lw=2))
    ax2.text(row['PC1'] * 1.12, row['PC2'] * 1.12,
             culture, fontsize=9, ha='center', color='#1E3A8A')

cercle = plt.Circle((0, 0), 1, color='gray', fill=False, linestyle='--', alpha=0.4)
ax2.add_patch(cercle)
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.4)
ax2.axvline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.4)
ax2.set_xlim(-1.3, 1.3)
ax2.set_ylim(-1.3, 1.3)
ax2.set_aspect('equal')
ax2.set_title('Cercle des corrélations — Contribution des cultures',
              fontsize=14, fontweight='bold')
ax2.set_xlabel(f'PC1 ({var_pc1:.1f}%)', fontsize=12)
ax2.set_ylabel(f'PC2 ({var_pc2:.1f}%)', fontsize=12)

plt.tight_layout()
plt.savefig('acp_cercle.png', dpi=150, bbox_inches='tight')
plt.show()

> **Comment lire le graphique ACP :**
> - **Pays proches** = profils agricoles similaires
> - **Pays éloignés** = profils très différents
> - **PC1 (axe horizontal)** = oppose les grands producteurs céréaliers aux autres
> - **PC2 (axe vertical)** = oppose les producteurs de sucre/racines aux céréaliers
>
> **Ce qu'on observe :** La Chine, l'Inde et les USA s'isolent très loin en raison de leurs volumes exceptionnels. La France et l'Allemagne se regroupent (profil européen céréalier similaire).

---
## 8. Conclusions finales

| Étape | Ce qu'on a appris |
|---|---|
| Exploration | 2,2M lignes, 259 pays, 172 cultures, 1961–2007 |
| Top 10 | Les céréales dominent la production mondiale |
| Blé | +172% de croissance en 46 ans (Révolution verte) |
| Maïs | Domination nord-américaine, montée de l'Asie après 1980 |
| ACP | Les grands producteurs (Chine, USA, Inde) ont des profils uniques |